In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anirudhcv/labeled-optical-coherence-tomography-oct")

print("Path to dataset files:", path)

100%|██████████| 6.70G/6.70G [01:09<00:00, 104MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2


In [ ]:
import os
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report, confusion_matrix


# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Path
root_dir = "/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2/Dataset - train+val+test"
train_dir = os.path.join(root_dir, "train")
val_dir   = os.path.join(root_dir, "val")
test_dir  = os.path.join(root_dir, "test")

# Custom OCT Dataset
class OCTDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.classes = sorted(os.listdir(root))
        self.samples = []
        for label, cls in enumerate(self.classes):
            cls_dir = os.path.join(root, cls)
            for img_name in os.listdir(cls_dir):
                img_path = os.path.join(cls_dir, img_name)
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_path, label = self.samples[index]
        img = Image.open(img_path).convert("L")   # grayscale
        if self.transform:
            img = self.transform(img)
        return img, label
# Data Augmentation
MEAN = [0.5]
STD  = [0.5]

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomAffine(
        degrees=7,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    T.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])
# DataLoader（mini-batch）

batch_size = 64
num_workers = 2
train_set = OCTDataset(train_dir, transform=train_transform)
val_set   = OCTDataset(val_dir,   transform=eval_transform)
test_set  = OCTDataset(test_dir,  transform=eval_transform)

num_classes = len(train_set.classes)
print("Classes:", train_set.classes)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)

# ResNet-style CNN with SE attention

class SEBlock(nn.Module):
  """Squeeze-and-Excitation channel attention"""
  def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

  def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class BasicBlock(nn.Module):
  """Simplified ResNet Basic Block with optional SE"""
  expansion = 1
  def __init__(self, in_channels, out_channels, stride=1, use_se=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.use_se = use_se
        if use_se:
            self.se = SEBlock(out_channels)

  def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        if self.use_se:
            out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = F.relu(out, inplace=True)
        return out

class ResOCTNet(nn.Module):
  """ResNet-like CNN optimized for OCT classification"""
  def __init__(self, num_classes=4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # 224 -> 56
        )
        self.layer1 = self._make_layer(32,   64, blocks=2, stride=1, use_se=True)
        self.layer2 = self._make_layer(64,  128, blocks=2, stride=2, use_se=True)
        self.layer3 = self._make_layer(128, 256, blocks=2, stride=2, use_se=True)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(256, num_classes)

  def _make_layer(self, in_ch, out_ch, blocks, stride, use_se):
        layers = [BasicBlock(in_ch, out_ch, stride=stride, use_se=use_se)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_ch, out_ch, stride=1, use_se=use_se))
        return nn.Sequential(*layers)

  def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# MixUp utilities
def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, preds, y_a, y_b, lam):
    return lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)

# EMA (Exponential Moving Average)
def create_ema_model(model):
    ema = ResOCTNet(num_classes=num_classes).to(device)
    ema.load_state_dict(model.state_dict())
    for p in ema.parameters():
        p.requires_grad_(False)
    return ema

@torch.no_grad()
def update_ema(ema_model, model, alpha=0.999):
    for ema_p, p in zip(ema_model.parameters(), model.parameters()):
        ema_p.data.mul_(alpha).add_(p.data, alpha=1 - alpha)

# Training and validation loops
model = ResOCTNet(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
scaler = GradScaler()
num_epochs = 20

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    return running_loss / total, correct / total


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    return running_loss / total, correct / total


best_val_acc = 0.0
best_model_path = "/content/best_resoctnet_simple.pth"

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
    val_loss,   val_acc   = eval_one_epoch(model, val_loader,   criterion, device)
    scheduler.step()

    print(f"Epoch [{epoch}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  New best model saved (val_acc = {best_val_acc:.4f})")

print("Training finished. Best val_acc:", best_val_acc)

Using device: cuda
Classes: ['CNV', 'DME', 'DRUSEN', 'NORMAL']


/tmp/ipython-input-2409081541.py:209: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-2409081541.py:222: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-2409081541.py:247: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/20] Train Loss: 0.3338 Acc: 0.8848 | Val Loss: 0.1915 Acc: 0.9373
  👉 New best model saved (val_acc = 0.9373)
Epoch [2/20] Train Loss: 0.1770 Acc: 0.9418 | Val Loss: 0.1628 Acc: 0.9457
  👉 New best model saved (val_acc = 0.9457)
Epoch [3/20] Train Loss: 0.1543 Acc: 0.9490 | Val Loss: 0.1301 Acc: 0.9576
  👉 New best model saved (val_acc = 0.9576)
Epoch [4/20] Train Loss: 0.1385 Acc: 0.9545 | Val Loss: 0.1082 Acc: 0.9627
  👉 New best model saved (val_acc = 0.9627)
Epoch [5/20] Train Loss: 0.1275 Acc: 0.9576 | Val Loss: 0.1175 Acc: 0.9619
Epoch [6/20] Train Loss: 0.1177 Acc: 0.9606 | Val Loss: 0.1008 Acc: 0.9663
  👉 New best model saved (val_acc = 0.9663)
Epoch [7/20] Train Loss: 0.1117 Acc: 0.9629 | Val Loss: 0.0990 Acc: 0.9661
Epoch [8/20] Train Loss: 0.1059 Acc: 0.9645 | Val Loss: 0.1089 Acc: 0.9645
Epoch [9/20] Train Loss: 0.0969 Acc: 0.9672 | Val Loss: 0.0900 Acc: 0.9695
  👉 New best model saved (val_acc = 0.9695)
Epoch [10/20] Train Loss: 0.0921 Acc: 0.9695 | Val Loss: 0.08

In [ ]:
# Load Best model for Test evaluation
best_model = ResOCTNet(num_classes=num_classes).to(device)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()

all_preds = []
all_labels = []

# Run test set inference
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = best_model(imgs)
        _, preds = outputs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

#  Classification report (Precision, Recall, F1)
print("\n Classification Report")
print(classification_report(all_labels, all_preds, target_names=train_set.classes, digits=4))

#  Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("\n Confusion Matrix")
print(cm)


 Classification Report
              precision    recall  f1-score   support

         CNV     0.9813    0.9786    0.9800      3746
         DME     0.9636    0.9569    0.9602      1161
      DRUSEN     0.9175    0.8906    0.9039       887
      NORMAL     0.9836    0.9920    0.9878      5139

    accuracy                         0.9755     10933
   macro avg     0.9615    0.9546    0.9580     10933
weighted avg     0.9753    0.9755    0.9754     10933


 Confusion Matrix
[[3666   17   59    4]
 [  23 1111    1   26]
 [  37    5  790   55]
 [  10   20   11 5098]]


In [ ]:
def predict_image(image_path):
    img = Image.open(image_path).convert("L")  # OCT images are grayscale
    img_t = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_t)
        prob = torch.softmax(outputs, dim=1)
        pred = torch.argmax(prob)

    print(f"Prediction: {classes[pred]}")
    print("Class probabilities:")
    for i, c in enumerate(classes):
        print(f"{c}: {prob[0][i].item():.4f}")

    return pred.item(), prob[0]

In [ ]:
predict_image("")